### Build and run MHCPrime on example datasets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntranoslab/MHCPrime/blob/main/notebooks/01_inference_quickstart.ipynb)

#### Setup notebook

In [ ]:
# set up directory structure and install mhcprime if running in colab
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)

if IN_COLAB:
    os.chdir("/content")

    if not Path("MHCPrime").exists():
        subprocess.run(
            ["git", "clone", "https://github.com/ntranoslab/MHCPrime.git"],
            check=True,
        )
    else:
        os.chdir("/content/MHCPrime")
        subprocess.run(["git", "pull"], check=True)
        os.chdir("/content")

    os.chdir("/content/MHCPrime")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "peptides",
            "transformers>=4.30,<4.58",
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-e",
            ".",
            "--no-deps",
        ],
        check=True,
    )

    REPO_ROOT = Path.cwd()

else:
    NOTEBOOK_DIR = Path.cwd()

    if NOTEBOOK_DIR.name == "notebooks":
        REPO_ROOT = NOTEBOOK_DIR.parent
    else:
        REPO_ROOT = NOTEBOOK_DIR

    os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Repo root:", REPO_ROOT)
print("src path added:", src_path)
print("src in sys.path:", src_path in sys.path)

In [ ]:
# imports
from pathlib import Path
import torch
import pandas as pd
from mhcprime import (
    load_example_dataset,
    build_mhcprime_model,
    load_mhcprime_model,
    load_default_mhc_pseudosequences,
    prepare_input_dataframe,
    predict_dataframe,
    print_data_stats,
    get_default_checkpoint_path
)

from mhcprime.metrics import *
from mhcprime.plotting import *

In [ ]:
# get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print(
        "No GPU detected. In Colab, go to Runtime → Change runtime type → "
        "Hardware accelerator → GPU, then restart and rerun."
    )

#### Get test dataframes and process

 - Required columns: `seq`, `allele`.
 - The allele should be formatted without colons or stars, for example, HLA-A*02:01 should be formatted as A0201.
 - If a given allele is not in the pseudosequence dictionary, it will be excluded, however, users can update the pseudosequence dictionary with their alleles, given that the selected positions match the remaining alleles.

In [ ]:
# small test data
test_data_small = load_example_dataset("small", index_col=0)
test_data_small_processed = prepare_input_dataframe(test_data_small)
print_data_stats(test_data_small_processed)
display(test_data_small_processed.head(10))

In [ ]:
# large test data
test_data_large = load_example_dataset("large", index_col=0)
test_data_large_processed = prepare_input_dataframe(test_data_large)
print_data_stats(test_data_large_processed)
display(test_data_large_processed.head(10))

In [ ]:
# large test data held at 1:99 pos to neg ratio
test_data_large_1_99 = load_example_dataset("large_1_99", index_col=0)
test_data_large_1_99_processed = prepare_input_dataframe(test_data_large_1_99)
print_data_stats(test_data_large_1_99_processed)
display(test_data_large_1_99_processed.head(10))

#### Load model and MHCPrime weights

In [ ]:
# get mhcprime without weights
model, tokenizer, model_params = build_mhcprime_model(
    device=device,
    print_params=True,
)

model.eval()

print("Model class:", type(model).__name__)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model device:", next(model.parameters()).device)
print("Number of model params:", sum(p.numel() for p in model.parameters()))

In [ ]:
# load trained mhcprime

print("Default checkpoint:", get_default_checkpoint_path())

base_model, tokenizer, model_params = load_mhcprime_model(
    device=device,
    strict=True,
    eval_mode=True,
    print_params=True,
)

print("Model device:", next(base_model.parameters()).device)
print("Training mode:", base_model.training)

#### Inference

 - The prediction wrapper accepts both processed and unprocessed dataframes. If entering in a preprocessed dataframe, use `preprocess = False`. The function by default outputs the unprocessed and unpadded dataframe, this can be changed with `return_processed`.
 - MHCPrime uses fast cached inference. The cache path specifies where the processed tensor cache is stored, and this cache can optionally be removed after inference.
 - On CUDA devices, fast cached inference uses bfloat16 mixed precision by default for efficient scoring. The output logits are returned as floating-point scores for downstream analysis.
 - The ranking dataframe is derived from the human proteome. The rank dataframe is also scored without flanks, as the base model was trained. We rank against a global background; this avoids scoring unseen alleles or pseudosequences when introduced to the model and aligns with how the base model was trained. The ranking dataframe can be altered, however, this requires you to score the inputted rank dataframe prior to ranking.
 - Fast cached inference performs a dtype conversion. To restore the original model dtype to run or train the model again after inference, use `restore_model_dtype = True`.
 - The raw scores are unbounded, however, we found a reasonable threshold around -100 to -50 for the base model. We suggest using ranks for threshold-based evaluation and raw scores for comparisons against a background/negative distribution.

In [ ]:
%%time
# cached inference on smaller dataframe
CACHE_PATH = REPO_ROOT / "dataset_cache.pt"

# returns unprocessed
test_data_small = predict_dataframe(
    model=base_model,
    df=test_data_small_processed,
    tokenizer=tokenizer,
    score_col="mhcprime",
    mode="fast",
    batch_size=3072,
    num_workers=8,
    device=device,
    preprocess=False,
    cache_path=CACHE_PATH,
    build_cache=True,
    remove_cache=True,
    use_bf16=True,
    restore_model_dtype=True,
    compile_model=False,
    disable_tqdm=False,
)

display(test_data_small[["seq", "allele", "label", "mhcprime", "mhcprime_rank"]].head(10))
print("Cache exists after inference:", CACHE_PATH.exists())

In [ ]:
%%time
# cached inference on larger dataframe
CACHE_PATH = REPO_ROOT / "dataset_cache.pt"

# returns unprocessed
test_data_large = predict_dataframe(
    model=base_model,
    df=test_data_large_processed,
    tokenizer=tokenizer,
    score_col="mhcprime",
    mode="fast",
    batch_size=3072,
    num_workers=8,
    device=device,
    preprocess=False,
    cache_path=CACHE_PATH,
    build_cache=True,
    remove_cache=True,
    use_bf16=True,
    restore_model_dtype=True,
    compile_model=False,
    disable_tqdm=False,
)

display(test_data_large[["seq", "allele", "label", "mhcprime", "mhcprime_rank"]].head(10))
print("Cache exists after inference:", CACHE_PATH.exists())

In [ ]:
%%time
# run the largest benchmarking dataset
CACHE_PATH = REPO_ROOT / "dataset_cache.pt"

# returns unprocessed df, so rename back to original
test_data_large_1_99 = predict_dataframe(
    model=base_model,
    df=test_data_large_1_99_processed,
    tokenizer=tokenizer,
    score_col="mhcprime",
    mode="fast",
    batch_size=3072,
    num_workers=8,
    device=device,
    preprocess=False,
    cache_path=CACHE_PATH,
    build_cache=True,
    remove_cache=True,
    use_bf16=True,
    restore_model_dtype=True,
    compile_model=False,
    disable_tqdm=False,
)

display(test_data_large_1_99[["seq", "allele", "label", "mhcprime", "mhcprime_rank"]].head(10))
print("Cache exists after inference:", CACHE_PATH.exists())

#### Benchmarking

 - Note, our benchmarking set is a small sample of the main test used to evaluate the MS performance on MHCPrime and other models. Full benchmarking should be performed on the full test set (packaged with the training set). 
 - Run analysis on `test_data_large_1_99` and output benchmarking results.

In [ ]:
# get ap matrix and pval df
score_cols=["mhcprime", "mhc_pred_0", "ms_log_odds", "score", "presentation_score", "EL_score", "BA_score", "munis", "transphla", "bigmhc", "deepattentionpan", "Pathogen_score_nmp42", "Neo_score_nmp42", "capsnetmhc_anthem", "capsnetmhc_iedb"]

# methods: "ap", "auc", "auc01", "ppvn", "spearman"
ms_allele_ap_matrix=compute_metric_matrix(test_data_large_1_99, score_cols, method="ap", disable_tqdm=False)

ms_allele_ap_matrix_pval_df=compute_pairwise_pvalues_general(test_data_large_1_99[score_cols])
ms_allele_ap_matrix_pval_df_converted=pval_to_stars(ms_allele_ap_matrix_pval_df)

In [ ]:
# ms performance boxplots
boxplot_groups={
    "MHCPrime": [(ms_allele_ap_matrix, 'mhcprime')],
    "HLApollo": [(ms_allele_ap_matrix, 'mhc_pred_0')],
    "MS+ LO": [(ms_allele_ap_matrix, 'ms_log_odds')],
    "MixMHCpred3.0": [(ms_allele_ap_matrix, 'score')],
    "MUNIS": [(ms_allele_ap_matrix, 'munis')],
    "BigMHC": [(ms_allele_ap_matrix, 'bigmhc')],
    "MHCFlurry2.0": [(ms_allele_ap_matrix, 'presentation_score')],
    "NetMHCpan4.1 EL": [(ms_allele_ap_matrix, 'EL_score')],
    "NetMHCpan4.2 NE": [(ms_allele_ap_matrix, 'Neo_score_nmp42')],
    "CapsNet-MHC Anthem": [(ms_allele_ap_matrix, 'capsnetmhc_anthem')],
    "NetMHCpan4.1 BA": [(ms_allele_ap_matrix, 'BA_score')],
    "DeepAttentionPan": [(ms_allele_ap_matrix, 'deepattentionpan')],
    "CapsNet-MHC IEDB": [(ms_allele_ap_matrix, 'capsnetmhc_iedb')],
    "TransPHLA": [(ms_allele_ap_matrix, 'transphla')],
    "NetMHCpan4.2 PS": [(ms_allele_ap_matrix, 'Pathogen_score_nmp42')],
     
}

transformer_based_color="#94B0F8"
nn_based_color="#ffce78"
lo_color="#bababa"
colors={
    "mhcprime": "#081470",
    "mhc_pred_0": transformer_based_color,
    "ms_log_odds": lo_color,
    "presentation_score":nn_based_color, 
    "score":nn_based_color, 
    "EL_score": nn_based_color,
    "BA_score": nn_based_color,
    "munis": transformer_based_color, 
    "transphla": transformer_based_color,
    "bigmhc": transformer_based_color, 
    "deepattentionpan": transformer_based_color,
    "Pathogen_score_nmp42":nn_based_color, 
    "Neo_score_nmp42":nn_based_color, 
    "capsnetmhc_anthem": transformer_based_color,
    "capsnetmhc_iedb": transformer_based_color, 
}

plot_params = {
    'title': '',
    'title_fontsize': 16,
    'title_bold': False,
    'xlabel': '',
    'ylabel': 'AP',
    'xlabel_fontsize': 12,
    'ylabel_fontsize': 12,
    'figsize': (10, 6),
    'dpi': 300,
    'legend_fontsize': 12,
    'legend_cols': 5,
    'legend_handle_size': 0.9,
    'marker_size': 3,
    'axis_linewidth': 0.5,
    'tick_length': 8.0,
    'xlabel_pad': 5.0,
    'ylabel_pad': 5.0,
    'xtick_labelsize': 10,
    'ytick_labelsize': 10,
    'xtick_rotation': 45

}

_=plot_grouped_boxplots(
    boxplot_groups,
    colors=colors,
    legend=None,
    plot_params=plot_params,
    grid_alpha=0.0,
    box_width=0.1,
    group_spacing=0.1,
    box_spacing=0.05,
    box_outline_color='black',
    box_outline_width=1.0,
    box_flier_marker='o',
    box_flier_size=3,
    box_flier_color="#2B2B2B",
    box_flier_alpha=0.6,
    box_flier_edge_width=0,
    box_median_color='black',
    box_median_width=1.5,
    box_whisker_style='-',
    box_whisker_width=1.0,
    box_notch=False,
    box_patch_alpha=1.0,
    hline_y=None,
    hline_label=None,
    hline_color='gray',
    hline_width=1.0,
    hline_style='--',
    hline_text=None,
    hline_text_position='above',
    hline_text_color=None,
    hline_text_fontsize=10,
    hline_text_offset=0.01,
    hline_text_x_offset=0.05,
    hline_in_legend=True,
    reference_hlines=None,
    legend_mode=None,
    legend_loc='lower center',
    legend_bbox=None,
    legend_ncol=None,
    ylim=(0, 1.0),
    compute_pvalues=False,
    pvalue_test='wilcoxon', 
    cluster_col='louvain_id',
    sort_by ="median",
    sort_ascending=False, 
    pval_df=ms_allele_ap_matrix_pval_df,
    pval_ref_col="mhcprime",
    pval_ref_stars_only=True,
    pval_format='stars',
    pval_fontsize=12,
    pval_color='black',
    pval_star_y=None,
    pval_star_y_frac=1.01,
    pval_star_text_offset=0.0,
    save_path=None,
)